In [4]:
# 加载数据集
from PIL import Image
import json
from pathlib import Path
import torch
from torchvision import transforms
import numpy as np

# 人脸数据集路径
dataset_dir = '/data1/humw/Datasets/VGGFace2'


def load_data(data_dir, image_size=512, resample=2):
    def image_to_numpy(image):
        return np.array(image).astype(np.uint8)

    images = []
    
    # 检查data_dir是单个图像路径还是目录路径
    if Path(data_dir).is_file():
        # 如果是单个图像路径，读取图像并复制4次
        image = Image.open(data_dir).convert("RGB")
        for _ in range(4):
            images.append(image_to_numpy(image))
    else:
        # 如果是目录路径，按原逻辑处理所有图像文件
        for i in list(Path(data_dir).iterdir()):
            if not i.suffix in [".jpg", ".png", ".jpeg"]:
                continue
            else:
                images.append(image_to_numpy(Image.open(i).convert("RGB")))
    
    # 调整图像大小
    images = [Image.fromarray(i).resize((image_size, image_size), resample) for i in images]
    
    # 转换为numpy数组并堆叠
    images = np.stack(images)
    
    # 调整维度顺序
    images = torch.from_numpy(images).permute(0, 3, 1, 2).float()
    
    # 断言图像尺寸的一致性
    assert images.shape[-1] == images.shape[-2]
    
    return images

train_aug = [
        transforms.Resize(512, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(512),
    ]
tensorize_and_normalize = [
    transforms.Normalize([0.5*255]*3,[0.5*255]*3),
]
all_trans = train_aug + tensorize_and_normalize
all_trans = transforms.Compose(all_trans)
    
# 加载模型
import torch
import os
import torch.nn.functional as F
from diffusers import AutoencoderKL

device = "cuda:0"
torch_dtype = torch.bfloat16
model = AutoencoderKL.from_pretrained(
            pretrained_model_name_or_path="/data1/humw/Pretrains/stable-diffusion-v1-5", 
            subfolder="vae", 
            revision="bf16"
        ).to(dtype=torch_dtype).eval().requires_grad_(False)
model = model.to(device)

ori_id_embeds_dict = {}
# 获取原始图像编码
person_id_list = sorted(os.listdir(dataset_dir))
for person_id in person_id_list:
    person_id_dir = os.path.join(dataset_dir, person_id, "set_B")
    clean_data = load_data(person_id_dir, 512, 2)
    original_data = clean_data.to(device).requires_grad_(False).to(dtype=torch_dtype)
    tran_original_data = all_trans(original_data).to(dtype=torch_dtype)
    ori_embeds = model.encode(tran_original_data).latent_dist.sample() * model.config.scaling_factor
    ori_id_embeds_dict[person_id] = ori_embeds
    
# 获取目标图像编码
target_images_dir = "/data1/humw/Codes/My-Anti-DreamBooth/target_images"
# 计算两两之间的编码余弦损失距离，距离越大越好
target_id_list = sorted(os.listdir(target_images_dir))
tgt_id_embeds_dict = {}
for target_id in target_id_list:
    target_image_path = os.path.join(target_images_dir, target_id)
    clean_data = load_data(target_image_path, 512, 2)
    original_data = clean_data.to(device).requires_grad_(False).to(dtype=torch_dtype)
    tran_original_data = all_trans(original_data).to(dtype=torch_dtype)
    ori_embeds = model.encode(tran_original_data).latent_dist.sample() * model.config.scaling_factor
    tgt_id_embeds_dict[target_id] = ori_embeds
    
max_id_map_id = dict()
max_id_map_loss = dict()
min_id_map_id = dict()
min_id_map_loss = dict()
random_id_map_id = dict()
random_id_map_loss = dict()
for person_id_i in person_id_list:
    
    max_id_map_id[person_id_i] = -1
    max_id_map_loss[person_id_i] = -2
    
    min_id_map_id[person_id_i] = -1
    min_id_map_loss[person_id_i] = 100000000
    
    random_idx = np.random.randint(0, len(target_id_list))
    random_id_map_id[person_id_i] = target_id_list[random_idx]
    
    for person_id_j in target_id_list:
        tmp = F.mse_loss(ori_id_embeds_dict[person_id_i], tgt_id_embeds_dict[person_id_j], reduction="mean") # mse损失的值应该大于0
        print(person_id_i, person_id_j, tmp)
        if tmp > max_id_map_loss[person_id_i]:
            max_id_map_id[person_id_i] = person_id_j
            max_id_map_loss[person_id_i] = tmp
        if tmp < min_id_map_loss[person_id_i]:
            min_id_map_id[person_id_i] = person_id_j
            min_id_map_loss[person_id_i] = tmp

n000050 baozheng_chisangzhen_black.png tensor(2.4688, device='cuda:0', dtype=torch.bfloat16)
n000050 caocao_qunyinghui_white.png tensor(2.3438, device='cuda:0', dtype=torch.bfloat16)
n000050 chengyaojin_jiajialou_green.png tensor(2.3125, device='cuda:0', dtype=torch.bfloat16)
n000050 dianwei_zhanwancheng_yellow.png tensor(2.4219, device='cuda:0', dtype=torch.bfloat16)
n000050 guanyu_huarongdao_red.png tensor(2.1875, device='cuda:0', dtype=torch.bfloat16)
n000050 lumeng_zoumaicheng_blue.png tensor(2.2812, device='cuda:0', dtype=torch.bfloat16)
n000050 pengyue_jiulishan_green.png tensor(2.1875, device='cuda:0', dtype=torch.bfloat16)
n000050 simayi_kongchengji_white.png tensor(2.4688, device='cuda:0', dtype=torch.bfloat16)
n000050 yangzhi_shengchengang_blue.png tensor(2.2188, device='cuda:0', dtype=torch.bfloat16)
n000050 yingbu_jiulishan_yellow.png tensor(2.4531, device='cuda:0', dtype=torch.bfloat16)
n000050 yuchigong_bailiangguan_black.png tensor(2.5156, device='cuda:0', dtype=torch.bf

In [5]:
print(max_id_map_id)
print(min_id_map_id)
print(random_id_map_id)

{'n000050': 'yuchigong_bailiangguan_black.png', 'n000057': 'baozheng_chisangzhen_black.png', 'n000058': 'baozheng_chisangzhen_black.png', 'n000061': 'baozheng_chisangzhen_black.png', 'n000063': 'baozheng_chisangzhen_black.png', 'n000068': 'baozheng_chisangzhen_black.png', 'n000076': 'simayi_kongchengji_white.png', 'n000080': 'yuchigong_bailiangguan_black.png', 'n000087': 'baozheng_chisangzhen_black.png', 'n000088': 'baozheng_chisangzhen_black.png', 'n000089': 'baozheng_chisangzhen_black.png', 'n000090': 'yuchigong_bailiangguan_black.png', 'n000097': 'baozheng_chisangzhen_black.png', 'n000098': 'baozheng_chisangzhen_black.png', 'n000103': 'baozheng_chisangzhen_black.png', 'n000104': 'baozheng_chisangzhen_black.png', 'n000105': 'baozheng_chisangzhen_black.png', 'n000110': 'baozheng_chisangzhen_black.png', 'n000138': 'baozheng_chisangzhen_black.png', 'n000139': 'baozheng_chisangzhen_black.png', 'n000142': 'baozheng_chisangzhen_black.png', 'n000145': 'baozheng_chisangzhen_black.png', 'n000

In [6]:
# 保存json文件
max_dict = max_id_map_id
json.dump(max_dict,open('/data1/humw/Codes/My-Anti-DreamBooth/max_vae_mse_VGGFace2-mask.json','w'), indent=4)

min_dict = min_id_map_id
json.dump(min_dict,open('/data1/humw/Codes/My-Anti-DreamBooth/min_vae_mse_VGGFace2-mask.json','w'), indent=4)

random_dict = random_id_map_id
json.dump(random_dict,open('/data1/humw/Codes/My-Anti-DreamBooth/random_vae_mse_VGGFace2-mask.json','w'), indent=4)